<a href="https://colab.research.google.com/github/arups330/ElitLab_MED_VQA/blob/main/Abdomen(closed)_WITHOUT_CoT_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Closed-question fine-tuning **WITHOUT CoT** — all models, sequential, auto-push to Hugging Face

**Steps**
1. Upload your dataset **.zip** to Colab (Files panel on the left → upload, or run the upload cell). It is extracted automatically and `closed_with_CoT.csv` is located inside it.
2. Only rows whose answer is **Yes / No** are used. Everything else is ignored.
3. Each model: load 4-bit → LoRA → train → push adapter to HF → free GPU → next model.
4. Models already on the Hub are skipped, so after a disconnect just re-run the last cell.

**Hugging Face setup (once)**
* Write token: https://huggingface.co/settings/tokens
* Colab 🔑 Secrets → add `HF_TOKEN` → enable *Notebook access*
* Click *Agree and access* on https://huggingface.co/google/gemma-4-E4B-it and https://huggingface.co/google/medgemma-4b-it
* Runtime → Change runtime type → **GPU**

In [1]:
!pip install -q "unsloth==2026.9.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199

In [2]:
from unsloth import FastVisionModel
import torch, gc, os

# HF token from Colab Secrets (🔑 icon in left sidebar -> add HF_TOKEN, enable notebook access)
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

HF_USERNAME = "Arup330"
DATASET_TAG = "Abdomen"

REPO_IDS = {
    "Gemma4-E4B": "google/gemma-4-E4B-it",
    "MedGemma-4B": "google/medgemma-4b-it",
    "Qwen2.5-VL-7B": "Qwen/Qwen2.5-VL-7B-Instruct",
    "Llama-11B": "unsloth/Llama-3.2-11B-Vision-Instruct",
    "Qwen3-VL-8B": "Qwen/Qwen3-VL-8B-Instruct"
}

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# ---------------------------------------------------------------------------
# 3. Upload zip -> extract -> find closed_with_CoT.csv -> keep Yes/No rows only
# ---------------------------------------------------------------------------
import os
import glob
import zipfile
import pandas as pd
from PIL import Image

EXTRACT_DIR = "/content/data"

# Use a zip already uploaded to /content (Files panel), otherwise open the upload dialog
zips = glob.glob("/content/*.zip")
if not zips:
    from google.colab import files
    uploaded = files.upload()
    zips = [f"/content/{n}" for n in uploaded.keys()]
ZIP_PATH = zips[0]
print("Using zip:", ZIP_PATH)

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)
print("Extracted to", EXTRACT_DIR)

# Locate closed_with_CoT.csv anywhere inside the extracted folder (case-insensitive)
csv_paths = [p for p in glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True)
             if os.path.basename(p).lower() == "closed_with_cot.csv"]
assert csv_paths, "closed_with_CoT.csv not found inside the zip"
print(f"Found {len(csv_paths)} CSVs:")
for p in csv_paths:
    print(" ", p)

frames = []
for csv_path in csv_paths:
    split_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    frames.append(df)

cot_df = pd.concat(frames, ignore_index=True)

# Keep ONLY closed questions whose answer is Yes / No -- ignore everything else
before = len(cot_df)
ans = cot_df["answer"].astype(str).str.strip().str.lower()
cot_df = cot_df[ans.isin(["yes", "no"])].copy()
cot_df["answer"] = cot_df["answer"].astype(str).str.strip().str.lower().map({"yes": "Yes", "no": "No"})
cot_df = cot_df.reset_index(drop=True)
print(f"Kept {len(cot_df)} / {before} Yes/No rows "
      f"(Yes={(cot_df['answer']=='Yes').sum()}, No={(cot_df['answer']=='No').sum()})")

IMG_COL = "image_file" if "image_file" in cot_df.columns else "img_name"

# Index every image inside the zip by file name, so images can live in any sub-folder
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".gif", ".webp")
IMG_INDEX = {os.path.basename(p): p
             for p in glob.glob(os.path.join(EXTRACT_DIR, "**", "*"), recursive=True)
             if p.lower().endswith(IMG_EXTS)}
print(f"Indexed {len(IMG_INDEX)} image files")

def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(str(img_name))
    candidates = [
        os.path.join(split_dir, str(img_name)),
        os.path.join(split_dir, flat_name),
        os.path.join(split_dir, str(img_name).replace("/", "_")),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    if flat_name in IMG_INDEX:
        return IMG_INDEX[flat_name]
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")

# sanity check: first image resolves
print("First image:", resolve_image_path(cot_df.iloc[0]["split_dir"], cot_df.iloc[0][IMG_COL]))

Using zip: /content/Abdomen_Train_with_CoT.zip
Extracted to /content/data
Found 1 CSVs:
  /content/data/Abdomen_Train_with_CoT/CT/train/closed_with_CoT.csv
Kept 133 / 150 Yes/No rows (Yes=70, No=63)
Indexed 92 image files
First image: /content/data/Abdomen_Train_with_CoT/CT/train/xmlab104_source.jpg


In [4]:
# ---------------------------------------------------------------------------
# 4. convert_to_conversation -- instruction = your closed-question prompt,
#    assistant target = the one-word ground-truth answer (Yes / No)
# ---------------------------------------------------------------------------
def systemPrompt(question: str) -> str:
    return f"""
Context:
- You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.

Objective:
- Answer the user's question by verifying whether it is supported by the visual evidence in the medical image.

Inputs:
- Question: {question}

You have to think step by step.
Instructions:
1. Examine the medical image carefully.
2. Independently determine the relevant visual findings before considering the CoT.
3. Compare your own observations with the provided CoT.
4. If the CoT is inconsistent with the image, disregard it.
5. Answer the question using the following evidence priority:
1. Medical image (highest priority)
2. User question
3. CoT (only if verified by the image)
6. Never fabricate findings or rely on assumptions.
7. If the image does not provide sufficient evidence to support a "Yes" answer, return "No."

Output Requirements:
- Output exactly one word.
- Do not provide explanations, punctuation, or additional text.

Valid outputs:
Yes
No

"""

def convert_to_conversation(sample):
    instruction = systemPrompt(sample["question"])
    image_path = resolve_image_path(sample["split_dir"], sample[IMG_COL])
    image = Image.open(image_path).convert("RGB")

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": image},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["answer"]}],
        },
    ]
    return {"messages": conversation}
pass

print("Converting rows to Unsloth chat format...")
converted_dataset = [convert_to_conversation(row) for _, row in cot_df.iterrows()]
print(f"Done. {len(converted_dataset)} examples ready.")
print("Example:", converted_dataset[0]["messages"])

Converting rows to Unsloth chat format...
Done. 133 examples ready.
Example: [{'role': 'user', 'content': [{'type': 'text', 'text': '\nContext:\n- You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.\n\nObjective:\n- Answer the user\'s question by verifying whether it is supported by the visual evidence in the medical image.\n\nInputs:\n- Question: Does the picture contain liver?\n\nYou have to think step by step.\nInstructions:\n1. Examine the medical image carefully.\n2. Independently determine the relevant visual findings before considering the CoT.\n3. Compare your own observations with the provided CoT.\n4. If the CoT is inconsistent with the image, disregard it.\n5. Answer the question using the following evidence priority:\n1. Medical image (highest priority)\n2. User question\n3. CoT (only if verified by the image)\n6. Never fabricate findings or rely on ass

In [6]:
# ---------------------------------------------------------------------------
# 5. Train ALL models one by one: load -> LoRA -> before check -> train ->
#    after check -> save -> push to HF -> free GPU -> next model
# ---------------------------------------------------------------------------
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
results = {}

for model_name, repo_id in REPO_IDS.items():
    HUB_REPO  = f"{HF_USERNAME}/{DATASET_TAG}_closed_noCoT_{model_name}_lora"
    LOCAL_DIR = f"{DATASET_TAG}_closed_noCoT_{model_name}_lora"

    # Skip if already trained + pushed (lets you re-run after a Colab disconnect)
    if api.repo_exists(HUB_REPO):
        print(f"\n[SKIP] {model_name} already on Hub: https://huggingface.co/{HUB_REPO}")
        results[model_name] = "skipped"
        continue

    print("\n" + "="*70)
    print(f"MODEL: {model_name}  ({repo_id})  ->  {HUB_REPO}")
    print("="*70)

    try:
        # -------------------------------------------------------------------
        # 1. Load model
        # -------------------------------------------------------------------
        model, tokenizer = FastVisionModel.from_pretrained(
            repo_id,
            load_in_4bit = True,
            use_gradient_checkpointing = "unsloth",
            token = HF_TOKEN,
        )

        # -------------------------------------------------------------------
        # 2. Attach LoRA adapters
        # -------------------------------------------------------------------
        model = FastVisionModel.get_peft_model(
            model,
            finetune_vision_layers=True,
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            r=16,
            lora_alpha=16,
            lora_dropout=0,
            bias="none",
            random_state=3407,
            use_rslora=False,
            loftq_config=None,
        )

        # -------------------------------------------------------------------
        # 5. Quick check BEFORE fine-tuning
        # -------------------------------------------------------------------
        FastVisionModel.for_inference(model)
        sample = cot_df.iloc[0]
        test_instruction = systemPrompt(sample["question"])
        test_image = Image.open(resolve_image_path(sample["split_dir"], sample[IMG_COL])).convert("RGB")

        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": test_instruction}
        ]}]
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        inputs = tokenizer(test_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
        output_ids = model.generate(**inputs, max_new_tokens=8, use_cache=True, do_sample=False)
        print("\nBEFORE fine-tuning, model output:")
        print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
        print("Ground truth:", sample["answer"])

        # -------------------------------------------------------------------
        # 6. Train with SFTTrainer + UnslothVisionDataCollator
        # -------------------------------------------------------------------
        FastVisionModel.for_training(model)

        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            data_collator=UnslothVisionDataCollator(model, tokenizer),
            train_dataset=converted_dataset,
            args=SFTConfig(
                per_device_train_batch_size=2,
                gradient_accumulation_steps=4,
                warmup_steps=5,
                # max_steps=30,                # uncomment for a quick test run
                num_train_epochs=1,
                learning_rate=2e-4,
                fp16=not is_bf16_supported(),
                bf16=is_bf16_supported(),
                logging_steps=5,
                optim="adamw_8bit",
                weight_decay=0.01,
                lr_scheduler_type="linear",
                seed=3407,
                output_dir=f"outputs/{model_name}",
                report_to="none",
                save_strategy="no",

                remove_unused_columns=False,
                dataset_text_field="",
                dataset_kwargs={"skip_prepare_dataset": True},
                dataset_num_proc=2,
                max_seq_length=2048,
            ),
        )

        trainer_stats = trainer.train()

        # -------------------------------------------------------------------
        # 7. Check AFTER fine-tuning -- same sample as before
        # -------------------------------------------------------------------
        FastVisionModel.for_inference(model)
        output_ids = model.generate(**inputs, max_new_tokens=8, use_cache=True, do_sample=False)
        print("\nAFTER fine-tuning, model output:")
        print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
        print("Ground truth:", sample["answer"])

        # -------------------------------------------------------------------
        # 8. Save the LoRA adapter locally, then push to Hugging Face Hub
        # -------------------------------------------------------------------
        model.save_pretrained(LOCAL_DIR)
        tokenizer.save_pretrained(LOCAL_DIR)
        print(f"\nSaved locally to ./{LOCAL_DIR}")

        model.push_to_hub(HUB_REPO, token=HF_TOKEN)
        tokenizer.push_to_hub(HUB_REPO, token=HF_TOKEN)
        print(f"Pushed -> https://huggingface.co/{HUB_REPO}")
        results[model_name] = "OK"

    except Exception as e:
        import traceback; traceback.print_exc()
        results[model_name] = f"FAILED: {e}"

    # -------------------------------------------------------------------
    # Free GPU memory before the next model
    # -------------------------------------------------------------------
    for v in ["trainer", "model", "tokenizer", "inputs", "output_ids"]:
        if v in globals(): del globals()[v]
    gc.collect()
    torch.cuda.empty_cache()

print("\n================ SUMMARY ================")
for k, v in results.items():
    print(f"{k:15s} {v}")


MODEL: Gemma4-E4B  (google/gemma-4-E4B-it)  ->  Arup330/Abdomen_closed_noCoT_Gemma4-E4B_lora
==((====))==  Unsloth 2026.9.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/tmp/ipykernel_2885/1424353915.py", line 31, in <cell line: 0>
    model, tokenizer = FastVisionModel.from_pretrained(
                       ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        repo_id,
        ^^^^^^^^
    ...<2 lines>...
        token = HF_TOKEN,
        ^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/unsloth/models/loader_utils.py", line 1891, in _wrapper
    return fn(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/unsloth/models/loader.py", line 1878, in from_pretrained
    model, tokenizer = FastBaseModel.from_pretrained(
                       ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        model_name = model_name,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    ...<41 lines>...
        **kwargs,
        ^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/unsloth_zoo/temporary_patches/moe_grouped_modulelist.py", line 390, in wrapper
    result = func(*args, **kwargs)
  File "/usr/local/


MODEL: MedGemma-4B  (google/medgemma-4b-it)  ->  Arup330/Abdomen_closed_noCoT_MedGemma-4B_lora
==((====))==  Unsloth 2026.9.7: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpitaba61q/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpitaba61q/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Pushed -> https://huggingface.co/Arup330/Abdomen_closed_noCoT_Qwen3-VL-8B_lora

================ SUMMARY ================
Gemma4-E4B      FAILED: CUDA out of memory. Tried to allocate 5.25 GiB. GPU 0 has a total capacity of 14.56 GiB of which 4.08 GiB is free. Including non-PyTorch memory, this process has 10.48 GiB memory in use. Of the allocated memory 10.22 GiB is allocated by PyTorch, and 117.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
MedGemma-4B     FAILED: expected scalar type BFloat16 but found Float
Qwen2.5-VL-7B   OK
Llama-11B       FAILED: CUDA out of memory. Tried to allocate 252.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 193.81 MiB is free. Including non-PyTorch memory, this process